# 04 Feature Engineering

This notebook converts the cohort and Sepsis-3 labels into leakage-safe hourly structured trajectories for multi-horizon prediction. The outputs are designed to support both classical baselines and deep temporal models.

## Design choices

- Features are aggregated at hourly resolution.
- Only measurements before the horizon-specific prediction time are retained.
- Positive stays are censored at `sepsis_onset_time - horizon`.
- Negative stays are censored at `OUTTIME - horizon`.
- A configurable trailing history window is kept for temporal modeling.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy

In [3]:
import pandas as pd

from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths
from src.data_processing.feature_engineering import (
    aggregate_events_hourly,
    build_hourly_feature_table,
    build_horizon_prediction_rows,
    extract_feature_measurements,
    summarize_horizon_rows,
)

In [4]:
config['feature_engineering']

{'hourly_window_hours': 1,
 'history_window_hours': 48,
 'min_history_hours': 6,
 'aggregate_functions': ['mean', 'min', 'max', 'last'],
 'chart_feature_keywords': {'heart_rate': ['heart rate'],
  'sbp': ['arterial blood pressure systolic',
   'non invasive blood pressure systolic',
   'systolic blood pressure'],
  'dbp': ['arterial blood pressure diastolic',
   'non invasive blood pressure diastolic',
   'diastolic blood pressure'],
  'map': ['mean arterial',
   'arterial pressure mean',
   'arterial bp mean',
   'nbp mean',
   'art mean',
   'bp cuff [mean]',
   'manual bp mean',
   'femoral abp mean'],
  'respiratory_rate': ['respiratory rate'],
  'temperature_c': ['temperature c', 'temp c'],
  'spo2': ['spo2', 'o2 saturation pulseoxymetry'],
  'glucose_chart': ['glucose']},
 'lab_feature_keywords': {'wbc': ['white blood cells', 'wbc'],
  'hemoglobin': ['hemoglobin'],
  'creatinine': ['creatinine'],
  'platelet': ['platelet'],
  'bilirubin': ['bilirubin'],
  'lactate': ['lactate'],


## Load cohort artifacts from Notebook 03

In [5]:
cohort_dir = paths['processed_data_dir'] / '03_cohort_construction'
cohort_path = cohort_dir / 'cohort.csv'
labels_path = cohort_dir / 'sepsis_labels.csv'
splits_path = cohort_dir / 'patient_splits.csv'

assert cohort_path.exists(), 'Cohort file missing. Run 03_cohort_construction first.'
assert labels_path.exists(), 'Label file missing. Run 03_cohort_construction first.'
assert splits_path.exists(), 'Split file missing. Run 03_cohort_construction first.'

cohort_df = pd.read_csv(cohort_path, parse_dates=['INTIME', 'OUTTIME', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'DOB', 'DOD'])
labels_df = pd.read_csv(labels_path, parse_dates=['suspected_infection_time', 'sepsis_onset_time'])
splits_df = pd.read_csv(splits_path)
cohort_df.shape, labels_df.shape, splits_df.shape

((52904, 19), (52904, 8), (52904, 4))

## Extract candidate structured measurements

In [6]:
measurements = extract_feature_measurements(
    extracted_dir=paths['extracted_data_dir'],
    cohort=cohort_df,
    chart_feature_keywords=config['feature_engineering']['chart_feature_keywords'],
    lab_feature_keywords=config['feature_engineering']['lab_feature_keywords'],
    chunksize=config['dataset']['chunksize'],
    low_memory=config['dataset']['low_memory'],
)
measurements['chart_events'].head(), measurements['lab_events'].head()

(   SUBJECT_ID  HADM_ID  ICUSTAY_ID           charttime      feature_name  \
 0          36   165660    241249.0 2134-05-12 13:00:00        heart_rate   
 1          36   165660    241249.0 2134-05-12 13:00:00               sbp   
 2          36   165660    241249.0 2134-05-12 13:00:00               dbp   
 3          36   165660    241249.0 2134-05-12 13:00:00  respiratory_rate   
 4          36   165660    241249.0 2134-05-12 13:00:00              spo2   
 
    value INTIME OUTTIME  
 0   86.0    NaT     NaT  
 1  137.0    NaT     NaT  
 2   72.0    NaT     NaT  
 3   21.0    NaT     NaT  
 4   93.0    NaT     NaT  ,
     SUBJECT_ID   HADM_ID           charttime feature_name  value  ICUSTAY_ID  \
 47           3  145834.0 2101-10-22 04:00:00  bicarbonate   15.0    211552.0   
 48           3  145834.0 2101-10-22 04:00:00    bilirubin    0.8    211552.0   
 49           3  145834.0 2101-10-22 04:00:00   creatinine    1.9    211552.0   
 50           3  145834.0 2101-10-22 04:00:00    

In [7]:
chart_hourly_df = aggregate_events_hourly(
    events=measurements['chart_events'],
    aggregate_functions=config['feature_engineering']['aggregate_functions'],
)
lab_hourly_df = aggregate_events_hourly(
    events=measurements['lab_events'],
    aggregate_functions=config['feature_engineering']['aggregate_functions'],
)
chart_hourly_df.shape, lab_hourly_df.shape

((6598796, 36), (598723, 44))

## Build the master hourly feature table

In [8]:
hourly_features_df = build_hourly_feature_table(
    cohort=cohort_df,
    chart_hourly=chart_hourly_df,
    lab_hourly=lab_hourly_df,
)
hourly_features_df = hourly_features_df.merge(splits_df, on=['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID'], how='left')
hourly_features_df.head()

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,hour,dbp__last,dbp__max,dbp__mean,dbp__min,glucose_chart__last,glucose_chart__max,...,wbc__min,INTIME,OUTTIME,age_at_icu_intime,GENDER,ETHNICITY,FIRST_CAREUNIT,LAST_CAREUNIT,hours_since_icu_admit,split
0,2,163353.0,243653.0,2138-07-17 20:00:00,NaN,NaN,NaN,NaN,72.0,72.0,...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,163353.0,243653.0,2138-07-17 21:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,163353.0,243653.0,2138-07-17 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,145834.0,211552.0,2101-10-20 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2101-10-20 19:10:11,2101-10-26 20:43:09,76.0,M,WHITE,MICU,MICU,-1.169722,test
4,3,145834.0,211552.0,2101-10-20 19:00:00,NaN,NaN,NaN,NaN,281.0,281.0,...,11.3,2101-10-20 19:10:11,2101-10-26 20:43:09,76.0,M,WHITE,MICU,MICU,-0.169722,test


## Create leakage-safe horizon datasets

In [9]:
horizon_rows = build_horizon_prediction_rows(
    hourly_features=hourly_features_df,
    labels=labels_df,
    horizons_hours=config['prediction']['horizons_hours'],
    history_window_hours=config['feature_engineering']['history_window_hours'],
    min_history_hours=config['feature_engineering']['min_history_hours'],
)
horizon_summary_df = summarize_horizon_rows(horizon_rows)
horizon_summary_df

,dataset_name,row_count,icu_stay_count,positive_stay_count,feature_column_count
0,horizon_6h,1268294,38447,1814,72
1,horizon_12h,1131670,36341,1305,72
2,horizon_24h,860480,26374,917,72


In [10]:
horizon_rows['horizon_6h'].head() if 'horizon_6h' in horizon_rows else None

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,hour,dbp__last,dbp__max,dbp__mean,dbp__min,glucose_chart__last,glucose_chart__max,...,ETHNICITY,FIRST_CAREUNIT,LAST_CAREUNIT,hours_since_icu_admit,split,sepsis_onset_time,sepsis3_label,prediction_time,hours_to_prediction,prediction_horizon_hours
0,3,145834.0,211552.0,2101-10-24 15:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,WHITE,MICU,MICU,91.830278,test,NaT,0.0,2101-10-26 14:43:09,47.719167,6
1,3,145834.0,211552.0,2101-10-24 16:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,WHITE,MICU,MICU,92.830278,test,NaT,0.0,2101-10-26 14:43:09,46.719167,6
2,3,145834.0,211552.0,2101-10-24 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,WHITE,MICU,MICU,93.830278,test,NaT,0.0,2101-10-26 14:43:09,45.719167,6
3,3,145834.0,211552.0,2101-10-24 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,WHITE,MICU,MICU,94.830278,test,NaT,0.0,2101-10-26 14:43:09,44.719167,6
4,3,145834.0,211552.0,2101-10-24 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,WHITE,MICU,MICU,95.830278,test,NaT,0.0,2101-10-26 14:43:09,43.719167,6


## Save reusable artifacts

In [11]:
output_dir = paths['processed_data_dir'] / '04_feature_engineering'
artifact_bundle = {
    'chart_events': measurements['chart_events'],
    'lab_events': measurements['lab_events'],
    'resolved_chart_itemids': measurements['chart_itemids'],
    'resolved_lab_itemids': measurements['lab_itemids'],
    'chart_hourly': chart_hourly_df,
    'lab_hourly': lab_hourly_df,
    'hourly_features': hourly_features_df,
    'horizon_summary': horizon_summary_df,
}
for dataset_name, dataset_df in horizon_rows.items():
    artifact_bundle[dataset_name] = dataset_df

saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

{'chart_events': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/04_feature_engineering/chart_events.csv',
 'lab_events': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/04_feature_engineering/lab_events.csv',
 'resolved_chart_itemids': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/04_feature_engineering/resolved_chart_itemids.csv',
 'resolved_lab_itemids': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/04_feature_engineering/resolved_lab_itemids.csv',
 'chart_hourly': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/04_feature_engineering/chart_hourly.csv',
 'lab_hourly': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/04_feature_engineering/lab_hourly.csv',
 'hourly_features': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/04_feature_engineering/hourly_features.csv',
 'horizon_summ

In [12]:
manifest_path = paths['manifests_dir'] / '04_feature_engineering_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='04_feature_engineering',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'horizon_summary': horizon_summary_df.to_dict(orient='records'),
        'hourly_feature_columns': [column for column in hourly_features_df.columns if '__' in column],
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/manifests/04_feature_engineering_manifest.json')

## Review checklist

Before training baselines, verify:

- Which features resolved to actual MIMIC item IDs?
- How many ICU stays survive each prediction horizon?
- Are positive examples still present at 24h?
- Are there obvious sparsity problems that need imputation strategy changes?
- Do the horizon datasets have the temporal depth needed for recurrent or transformer models?

## Next notebook

`05_text_processing.ipynb` will create the temporally aligned clinical-note pipeline so we can pair note-derived embeddings with these structured trajectories.